# Credit Scoring Model — CodeAlpha Machine Learning Internship

**Task:** Credit Scoring Model  
**Objective:** Predict an applicant's creditworthiness using financial history.  
**Algorithms:** Logistic Regression, Decision Tree, Random Forest  
**Evaluation:** Precision, Recall, F1-score, ROC-AUC, confusion matrix, ROC curve

> **Dataset note:** `credit_scoring_dataset.csv` is a synthetic educational dataset generated for this project. It is not real customer or bank data, and results must not be used for actual lending decisions. The task brief mentions financial features such as income, debts, and payment history but does not name a mandatory CSV dataset. If your mentor provides an official dataset, replace this CSV and adjust the target/column names accordingly.

## Project workflow
1. Load and inspect data
2. Clean data and perform EDA
3. Engineer financial features
4. Split data into train/test sets
5. Preprocess numeric features
6. Train and compare classification models
7. Evaluate with required metrics
8. Interpret feature importance and summarize limitations


## 1. Import libraries

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, classification_report, ConfusionMatrixDisplay,
    RocCurveDisplay
)

RANDOM_STATE = 42
pd.set_option("display.max_columns", None)


## 2. Load dataset

In [ ]:
# Keep credit_scoring_dataset.csv in the same folder as this notebook.
df = pd.read_csv("credit_scoring_dataset.csv")

print("Dataset shape:", df.shape)
display(df.head())


## 3. Data quality checks

In [ ]:
print("Column names:")
print(df.columns.tolist())

print("\\nData types and non-null counts:")
df.info()

print("\\nMissing values per column:")
display(df.isnull().sum().to_frame("missing_count"))

print("Duplicate rows:", df.duplicated().sum())
print("\\nTarget distribution:")
display(df["creditworthy"].value_counts().rename(index={0: "Not creditworthy", 1: "Creditworthy"}))


## 4. Exploratory Data Analysis (EDA)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

df["creditworthy"].value_counts().sort_index().plot(
    kind="bar", ax=axes[0], title="Target Class Distribution"
)
axes[0].set_xlabel("creditworthy (0 = No, 1 = Yes)")
axes[0].set_ylabel("Number of applicants")

axes[1].scatter(df["annual_income"], df["existing_debt"], alpha=0.35)
axes[1].set_title("Annual Income vs Existing Debt")
axes[1].set_xlabel("Annual income")
axes[1].set_ylabel("Existing debt")

plt.tight_layout()
plt.show()

display(df.groupby("creditworthy")[[
    "annual_income", "existing_debt", "loan_amount",
    "late_payments_12m", "missed_payments_12m", "payment_history_score"
]].mean().round(2))


## 5. Feature engineering

In [ ]:
# Create useful ratios/features from financial history.
data = df.copy()

data["debt_to_income_ratio"] = (
    (data["existing_debt"] + data["loan_amount"]) /
    data["annual_income"].replace(0, np.nan)
)
data["monthly_income"] = data["annual_income"] / 12
data["expense_to_income_ratio"] = (
    data["monthly_expenses"] / data["monthly_income"].replace(0, np.nan)
)
data["total_payment_issues_12m"] = (
    data["late_payments_12m"] + data["missed_payments_12m"]
)

# customer_id is an identifier, not a predictive feature.
X = data.drop(columns=["creditworthy", "customer_id"])
y = data["creditworthy"]

print("Feature count after engineering:", X.shape[1])
display(X.head())


## 6. Train/test split and preprocessing

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)

numeric_features = X.select_dtypes(include=["number"]).columns.tolist()

numeric_preprocessor = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

preprocessor = ColumnTransformer(transformers=[
    ("numeric", numeric_preprocessor, numeric_features)
])

print("Training rows:", X_train.shape[0])
print("Testing rows:", X_test.shape[0])


## 7. Train classification models

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=2000, class_weight="balanced"),
    "Decision Tree": DecisionTreeClassifier(
        max_depth=6, min_samples_leaf=10, class_weight="balanced",
        random_state=RANDOM_STATE
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=250, max_depth=10, min_samples_leaf=5,
        class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1
    )
}

pipelines = {}
results = []

for model_name, model in models.items():
    pipe = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("model", model)
    ])
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    y_prob = pipe.predict_proba(X_test)[:, 1]

    pipelines[model_name] = pipe
    results.append({
        "Model": model_name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, zero_division=0),
        "Recall": recall_score(y_test, y_pred, zero_division=0),
        "F1-Score": f1_score(y_test, y_pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_test, y_prob)
    })

results_df = pd.DataFrame(results).sort_values("ROC-AUC", ascending=False)
display(results_df.round(4))


## 8. Compare model performance

In [ ]:
metrics = ["Precision", "Recall", "F1-Score", "ROC-AUC"]
results_df.set_index("Model")[metrics].plot(kind="bar", figsize=(11, 5), ylim=(0, 1.05))
plt.title("Model Comparison on the Test Set")
plt.ylabel("Score")
plt.xticks(rotation=15, ha="right")
plt.legend(loc="lower right")
plt.tight_layout()
plt.show()


## 9. Detailed evaluation of the best model

In [ ]:
best_model_name = results_df.iloc[0]["Model"]
best_model = pipelines[best_model_name]
best_pred = best_model.predict(X_test)
best_prob = best_model.predict_proba(X_test)[:, 1]

print("Best model by ROC-AUC:", best_model_name)
print("\\nClassification report:")
print(classification_report(
    y_test, best_pred,
    target_names=["Not creditworthy (0)", "Creditworthy (1)"],
    zero_division=0
))

ConfusionMatrixDisplay.from_predictions(
    y_test, best_pred,
    display_labels=["Not creditworthy", "Creditworthy"],
    cmap=None
)
plt.title(f"Confusion Matrix — {best_model_name}")
plt.tight_layout()
plt.show()

RocCurveDisplay.from_predictions(y_test, best_prob)
plt.title(f"ROC Curve — {best_model_name}")
plt.tight_layout()
plt.show()

print("ROC-AUC:", round(roc_auc_score(y_test, best_prob), 4))


## 10. Feature importance / model interpretation

In [ ]:
fitted_model = best_model.named_steps["model"]

if hasattr(fitted_model, "feature_importances_"):
    importance = pd.Series(
        fitted_model.feature_importances_, index=numeric_features
    ).sort_values(ascending=False).head(12)
    importance.sort_values().plot(kind="barh", figsize=(9, 5))
    plt.title(f"Top Feature Importances — {best_model_name}")
    plt.xlabel("Importance")
    plt.tight_layout()
    plt.show()
elif hasattr(fitted_model, "coef_"):
    coef = pd.Series(
        fitted_model.coef_[0], index=numeric_features
    ).sort_values(key=abs, ascending=False).head(12)
    coef.sort_values().plot(kind="barh", figsize=(9, 5))
    plt.title("Largest Logistic Regression Coefficients")
    plt.xlabel("Standardized coefficient")
    plt.tight_layout()
    plt.show()
else:
    print("This model does not expose built-in feature importance.")


## 11. Try a sample prediction

In [ ]:
# Demonstration only. This is not a real lending recommendation.
sample_applicant = X_test.iloc[[0]]
predicted_class = int(best_model.predict(sample_applicant)[0])
predicted_probability = float(best_model.predict_proba(sample_applicant)[0, 1])

print("Predicted class:", "Creditworthy" if predicted_class == 1 else "Not creditworthy")
print(f"Model-estimated probability for class 1: {predicted_probability:.2%}")
display(sample_applicant)


## 12. Conclusion

- Compared Logistic Regression, Decision Tree, and Random Forest on the same held-out test set.
- Evaluated Precision, Recall, F1-score, and ROC-AUC as requested in the task brief.
- Added financial feature engineering and visualizations for model comparison and interpretation.
- The best model is selected by test ROC-AUC in this notebook; report the actual values produced when you run it.

### Limitations and responsible use
This is an educational demonstration using synthetic data. Synthetic patterns do not represent real lending populations. Real credit decisions require validated, representative data, fairness/bias checks, explainability, privacy controls, regulatory compliance, and human oversight. Do not use this model to make actual financial decisions.

### Suggested GitHub repository
`CodeAlpha_CreditScoringModel`

**Repository description:** Machine learning creditworthiness classification project using financial features, Logistic Regression, Decision Tree, Random Forest, and evaluation metrics including Precision, Recall, F1-score, and ROC-AUC.
